# 02｜第 2 節：交叉驗證與集成學習

**必做只有 4 組練習。** 每組看上方 Markdown 範例，再於空白 Code Cell 練習。

**必做流程：讀取 CSV → 選擇 X／y 並切分 → One-Hot → 隨機森林 `fit()`、`predict()`、評估。**

五折交叉驗證、Grid Search、AdaBoost 與新郵件預測改為 **Optional（選做）**，可以直接執行。

本節與第 1 節**可獨立執行**。先將 CSV 放在 Colab `/content` 或 Notebook 的工作目錄。


In [ ]:
# Import all libraries used in this notebook.
# 一次匯入本份 Notebook 需要的套件。
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

# Use a Chinese font where it is installed.
# 如果電腦有安裝微軟正黑體，就使用這個字型。
matplotlib.rc('font', family='Microsoft JhengHei')

# Fix minus signs on chart axes.
# 避免負號顯示為方框。
matplotlib.rcParams['axes.unicode_minus'] = False

# Use the built-in font in Google Colab.
# Colab 通常沒有微軟正黑體，因此使用內建字型。
if Path('/content').exists():
    matplotlib.rc('font', family='DejaVu Sans')

## 資料集欄位（500 封模擬校園郵件）

| 欄位 | 內容 | 用途 |
|---|---|---|
| `email_id` | 郵件編號 | 識別用，不放入模型 |
| `link_count` | 網址連結數量 | 數值特徵 X |
| `urgent_words` | 急迫用語出現次數 | 數值特徵 X |
| `exclamation_count` | 驚嘆號數量 | 數值特徵 X |
| `sender_known` | 是否為已知寄件者（1＝是） | 0／1 特徵 X |
| `email_type` | 郵件類型 | 類別特徵 X（One-Hot） |
| `is_spam` | 1＝垃圾郵件；0＝正常郵件 | 正確答案 y |

本份資料 **沒有缺漏值**，不需要使用 `fillna()`。兩節都使用這 500 筆資料，但可以各自獨立執行。


## 必做：4 組練習


### 練習 1／4｜讀取資料

使用相同的郵件資料集，先看前五筆。

**參考程式：**

```python
# Read the CSV file.
# 讀取 CSV 檔案。
df = pd.read_csv('02_校園郵件分類_500筆.csv')

# View the first five emails.
# 查看前五筆郵件資料。
display(df.head())
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 2／4｜選擇 X／y，切分資料

先保留 20% 的測試資料。後面的交叉驗證（Optional）也只使用訓練集。

**參考程式：**

```python
# Separate the input features and correct answers.
# 區分輸入特徵 X 與正確答案 y。
X = df.drop(columns=['email_id', 'is_spam'])
y = df['is_spam']

# Keep 20% of the data for testing.
# 保留 20% 的郵件作為測試資料。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Show the number of training and test emails.
# 顯示訓練集與測試集各有幾封郵件。
print('Train:', len(X_train), 'Test:', len(X_test))
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 3／4｜One-Hot：準備模型輸入

隨機森林與決策樹**通常不用標準化**。這份資料沒有缺漏值，因此只需把郵件類型轉成 0／1 欄位。

**參考程式：**

```python
# Convert the email types into 0/1 columns.
# 將郵件類型轉成 0／1 欄位。
X_train_ready = pd.get_dummies(X_train, dtype=int)
X_test_ready = pd.get_dummies(X_test, dtype=int)

# View the data before training.
# 查看模型即將使用的特徵。
display(X_train_ready.head())
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 4／4｜隨機森林：訓練、預測、評估

Random Forest 是由多棵決策樹組成的分類模型。使用 `fit()` 訓練、`predict()` 預測，再計算 Accuracy 與 F1。

**參考程式：**

```python
# Create a random forest model.
# 建立隨機森林分類模型。
model = RandomForestClassifier(n_estimators=60, max_depth=5, random_state=42)

# Train the model using the training data.
# 用訓練集訓練模型。
model.fit(X_train_ready, y_train)

# Predict labels for the test emails.
# 預測測試集是否為垃圾郵件。
pred = model.predict(X_test_ready)

# Check the prediction performance.
# 查看預測成效。
print('Accuracy:', round(accuracy_score(y_test, pred), 2))
print('F1:', round(f1_score(y_test, pred), 2))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 必做完成：你已經做出隨機森林模型

這一節完成了 **CSV → One-Hot → `fit()` → `predict()` → Accuracy／F1** 的完整流程。

以下都是 **Optional（選做）**：直接執行程式，觀察更進階的模型比較結果；若時間不夠，可以留待課後。


### Optional A｜五折交叉驗證（Cross-validation）

把**訓練集**輪流分成五份來驗證，得到五個 F1 分數，再比較決策樹與隨機森林的平均表現。這裡不使用測試集選擇模型。


In [ ]:
# Create a simple decision tree for comparison.
# 建立一棵決策樹，用來和隨機森林比較。
tree = DecisionTreeClassifier(max_depth=3, random_state=42)

# Calculate five validation scores for each model.
# 分別計算兩個模型的五折驗證 F1。
tree_cv = cross_val_score(tree, X_train_ready, y_train, cv=5, scoring='f1')
forest_cv = cross_val_score(model, X_train_ready, y_train, cv=5, scoring='f1')

# Compare average validation scores.
# 比較兩個模型的平均驗證 F1。
print('Tree CV F1:', round(tree_cv.mean(), 2))
print('Forest CV F1:', round(forest_cv.mean(), 2))

# Display the score for each validation fold.
# 畫出每一折的驗證結果。
plt.plot([1, 2, 3, 4, 5], tree_cv, marker='o', label='Tree')
plt.plot([1, 2, 3, 4, 5], forest_cv, marker='o', label='Forest')
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel('Fold')
plt.ylabel('F1')
plt.ylim(0, 1)
plt.legend()
plt.show()

### Optional B｜Grid Search：比較 SVM 的不同參數

自動在訓練集內以五折交叉驗證比較 `C`、`gamma`，看哪組 F1 較好。這是延伸示範；其他資料若需要標準化，應放進 CV Pipeline，避免資料洩漏。


In [ ]:
# Try different SVM parameter combinations.
# 搜尋不同的 SVM 參數組合。
grid = GridSearchCV(SVC(kernel='rbf'), {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]}, cv=5, scoring='f1')

# Choose parameters using training data only.
# 僅使用訓練集進行交叉驗證與參數選擇。
grid.fit(X_train_ready, y_train)

# Show the selected parameters and validation F1.
# 顯示選出的參數與驗證 F1。
print('Best parameters:', grid.best_params_)
print('Best CV F1:', round(grid.best_score_, 2))

### Optional C｜AdaBoost：比較另一種集成方法

AdaBoost 也是結合多個模型的方法。直接執行並比較它和隨機森林的五折 F1。


In [ ]:
# Create an AdaBoost classifier.
# 建立 AdaBoost 分類模型。
boost = AdaBoostClassifier(n_estimators=40, random_state=42)

# Compare both models using training data only.
# 使用訓練資料比較兩種模型的五折驗證 F1。
boost_cv = cross_val_score(boost, X_train_ready, y_train, cv=5, scoring='f1')
forest_cv = cross_val_score(model, X_train_ready, y_train, cv=5, scoring='f1')

# Print the two validation results.
# 顯示兩個模型的平均驗證 F1。
print('Random Forest F1:', round(forest_cv.mean(), 2))
print('AdaBoost F1:', round(boost_cv.mean(), 2))

# Plot the model comparison.
# 使用長條圖比較模型成效。
plt.bar(['Forest', 'AdaBoost'], [forest_cv.mean(), boost_cv.mean()])
plt.ylim(0, 1)
plt.ylabel('Cross-validation F1')
plt.show()

### Optional D｜預測四封新郵件

沿用已經訓練好的隨機森林。這裡準備四封不同類型的郵件，讓 One-Hot 編碼產生與訓練資料相同的欄位，可以直接預測。

**注意：** 這是教學用的簡化方式。真正的新資料若缺少某些郵件類型，仍須用更完整的方法處理欄位一致性。


In [ ]:
# Create four example emails.
# 輸入四封不同類型的模擬郵件。
new_data = pd.DataFrame({
    'link_count': [0, 2, 5, 1],
    'urgent_words': [0, 2, 4, 0],
    'exclamation_count': [0, 2, 4, 1],
    'sender_known': [1, 0, 0, 1],
    'email_type': ['校務公告', '社團活動', '優惠資訊', '課程通知']
})

# Convert the email types into 0/1 columns.
# 將新郵件類型轉成 0／1 欄位。
X_new_ready = pd.get_dummies(new_data, dtype=int)

# Predict the new emails using the trained forest.
# 使用已訓練的隨機森林預測新郵件。
new_data['預測（1=垃圾）'] = model.predict(X_new_ready)

# Display the predictions.
# 顯示四封新郵件的預測結果。
display(new_data)

**第 2 節完成。** 完整流程是核心；五折交叉驗證與其他模型比較可依進度選做。
